# Meridian Components Ltd — Variance Analysis Demo
### PyFi | Python + AI for Finance

---

**The business problem.** Meridian's FP&A team manages two manufacturing plants — Plant A (precision aerospace parts, ~$32M revenue) and Plant B (industrial components, ~$22M revenue). Every month, the finance team spends the better part of three days producing the management accounts pack. The bulk of that time is variance commentary — pulling actuals from the ERP, reconciling to budget across 76 line items, writing explanations, summarising by department, drafting the CFO brief. By the time it's done, the numbers have moved on.

**What this demo shows.** Python compresses that three-day process to minutes — not by being smarter than a finance professional, but by removing them from the loop for the parts that don't require their judgment. The analysis runs the same way every month, with one function call, producing the same quality of output regardless of who's in the office.

**The three things this demo proves Python can do that Excel + ChatGPT cannot:**
1. **Automation** — `analysis.run()` executes the full pipeline in one call, every month, no human required
2. **Async at scale** — 171 API calls fire concurrently via `asyncio.gather()`. A human doing this in ChatGPT makes 171 sequential copy-paste operations. Python makes them simultaneously.
3. **Governed context** — Python constructs every prompt precisely: company name, plant, department, line item, budget, actual, variance, direction. The model does a specific finance job reliably. ChatGPT in a browser does a general job approximately.

---

## Setup

In [ ]:
%pip install .. -q

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your key

---
## Step 1 — The raw data

Two CSV files. One per plant. That's the only monthly input. The whole pipeline works from this.

In [ ]:
import pandas as pd

plant_a = pd.read_csv("data/input/plant_a.csv")
plant_b = pd.read_csv("data/input/plant_b.csv")

print(f"Plant A: {len(plant_a)} line items across {plant_a['department'].nunique()} departments")
print(f"Plant B: {len(plant_b)} line items across {plant_b['department'].nunique()} departments")
print(f"Total:   {len(plant_a) + len(plant_b)} line items")
plant_a.head(6)

---
## Step 2 — The variance engine

Here is a rule that Excel gets wrong silently. Revenue and cost lines are directionally opposite — a favourable revenue variance means actual *exceeded* budget, but a favourable cost variance means actual came in *below* budget. Python encodes this once, correctly, and applies it to all 76 lines without error.

In [ ]:
from analysis.run.pipeline import load, compute_variances

df = load()
df = compute_variances(df)

# Show sign convention working correctly across revenue and cost lines
cols = ["plant", "department", "line_item", "budget", "actual", "variance_dollars", "is_favourable"]
df[cols].head(14)

In [ ]:
# Instant ranking — worst variances across the whole group
worst = df[~df["is_favourable"]].nsmallest(8, "variance_dollars")
worst[["plant", "department", "line_item", "variance_dollars", "variance_pct"]]

---
## Step 3 — The governed AI pipeline

Before running at scale, look at exactly what gets sent to the model for a single line. This is what 'governed AI' means in practice.

In [ ]:
from analysis.run.pipeline import _commentary_system, _commentary_user, _severity_system, _severity_user

# Pick a high-variance line to make the demo compelling
sample = df[df["line_item"] == "Raw Materials – Steel & Titanium Alloys"].iloc[0]

print("── SYSTEM PROMPT (sent once, controls all output) ──────────────────")
print(_commentary_system())
print()
print("── USER PROMPT (one per line item, constructed by Python) ──────────")
print(_commentary_user(sample))

In [ ]:
# The severity classifier — a separate concurrent call per line
print("── SEVERITY CLASSIFIER PROMPT ──────────────────────────────────────")
print(_severity_system())
print()
print(_severity_user(sample))

**Notice what Python is doing here.** It's not just sending text to an AI. It's constructing a precisely formatted prompt for each of 76 line items, with the correct directional language, the correct company context, and output constraints baked in. The model has no room to hallucinate direction or format — Python has already encoded those rules.

Now watch what happens when we fire all of these at once.

---
## Step 4 — Run the full pipeline

**171 API calls. Three passes. All concurrent.**

- Pass 1: 76 line-item commentary calls
- Pass 2: 76 severity classification calls + 16 department summary calls (92 simultaneous)
- Pass 3: 2 plant executive briefs + 1 consolidated CFO brief

A finance analyst doing this manually in ChatGPT would spend the better part of a day on 171 sequential prompts. Python fires them concurrently and returns results in roughly the time it takes to make one.

In [ ]:
import analysis

results = analysis.run()

In [ ]:
# The enriched DataFrame
df_out = results["df"]
df_out[["plant", "department", "line_item", "variance_dollars", "ai_severity", "ai_commentary"]].head(16)

In [ ]:
# Department summaries — synthesised from the line items, also generated by AI
dept_summaries = results["dept_summaries"]

# Show Plant A Cost of Sales summary
print(dept_summaries[("Plant A", "Cost of Sales")])

In [ ]:
# The consolidated CFO brief — generated from the plant summaries
print(results["cfo_brief"])

---
## Step 5 — Conversational analysis

The `Chat` class injects the full context — all 76 line items, all department summaries, both plant briefs, the CFO brief — into every session automatically. Python builds the context string; the analyst asks questions. No pasting required.

In [ ]:
from analysis import Chat

chat = Chat(
    df=results["df"],
    dept_summaries=results["dept_summaries"],
    plant_briefs=results["plant_briefs"],
    cfo_brief=results["cfo_brief"],
)

In [ ]:
chat.msg("Compare raw material performance across Plant A and Plant B. Which plant has the bigger problem?")

In [ ]:
chat.msg("Which three line items should the CFO prioritise in Monday's review and why?")

In [ ]:
chat.msg("If we bring Plant A raw material costs back to budget, what does the group gross profit look like?")

In [ ]:
# Your question here
chat.msg("")

---
## What you just saw

| | Manual (Excel + ChatGPT) | Python pipeline |
|---|---|---|
| Line-item commentary | 76 copy-paste prompts | 76 concurrent API calls |
| Severity classification | Manual judgement per line | 76 concurrent classifier calls |
| Department summaries | Written by hand | 16 concurrent calls |
| Plant briefs + CFO brief | Senior analyst, hours | 3 concurrent calls |
| Monthly effort | Repeat from scratch | `analysis.run()` |
| Context for chat | Paste table into browser | Injected automatically |

**That is the difference between using AI as a tool and building AI as infrastructure.**